# Análisis Exploratorio de Datos: Uso del Suelo a lo largo del Tiempo (2011, 2014, 2021)

Este notebook analiza la evolución del uso del suelo y el Índice Ambiental Ponderado (WEI) para los 21 municipios de Cundinamarca en los años 2011, 2014 y 2021.

Utilizamos las matrices de transición de uso del suelo para los periods 2011-2014, 2014-2017 y 2018-2021 para calcular:
- Área por clase de uso del suelo (Forest formation, Agricultural and livestock area, Non-vegetated area, Water body)
- Porcentajes de cada clase
- Índice Ambiental Ponderado (WEI)

Luego, creamos gráficos interactivos de líneas con Plotly donde:
- Eje X: años (2011, 2014, 2021)
- Eje Y: valor (área, porcentaje o WEI)
- Cada municipio es una línea (diferenciada por color)
- Cada clase es un tipo de línea (continuo, punteado, etc.)
- Incluimos casillas de verificación para mostrar/ocultar municipios y clases


In [30]:
# Importaciones y configuración inicial
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys
import re
import unicodedata

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from ipywidgets import Checkbox, VBox, HBox, Label, Layout, Dropdown, Output
from IPython.display import display, clear_output

# Añadir el directorio raíz al path para importar módulos locales
cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == 'src' else cwd
sys.path.append(str(root / 'src'))

# Importar funciones auxiliares del análisis existente
from soil_poverty_analysis import (
    find_repo_root,
    parse_weight_series,
    normalize_municipality,
)

# Encontrar la raíz del repositorio
root = find_repo_root(root)
DATA_DIR = root / 'data'
TRANSITIONS_DIR = DATA_DIR / 'transitions'

# Configuración de estilo para gráficos
import plotly.io as pio
pio.templates.default = "simple_white"

In [31]:
# Funciones auxiliares (copiadas del notebook 30_transicion_pobreza_dispersion_adj.ipynb)
manual_mapping = {
    'bojaca': 'Bojacá',
    'cajica': 'Cajicá',
    'chia': 'Chía',
    'cota': 'Cota',
    'facatativa': 'Facatativá',
    'funza': 'Funza',
    'gachancipa': 'Gachancipá',
    'sibate': 'Sibaté',
    'sopo': 'Sopó',
    'tocancipa': 'Tocancipá',
    'zipacon': 'Zipacón',
    'zipaquira': 'Zipaquirá',
    'el rosal': 'El Rosal',
    'la calera': 'La Calera',
    'madrid': 'Madrid',
    'mosquera': 'Mosquera',
    'subachoque': 'Subachoque',
    'tabio': 'Tabio',
    'tenjo': 'Tenjo',
    'soacha': 'Soacha',
}

subregion_mapping = {'Guavio': ['La Calera'],
                     'Sabana Centro': ['Cajicá ', 'Chía ', 'Cota', 'Sopó ', 'Tabio ', 'Tenjo'],
                     'Sabana Norte': ['Gachancipá  ', 'Tocancipá  ', 'Zipaquirá'],
                     'Sabana noroccidente': ['El Rosal', 'Facatativá ', 'Subachoque'],
                     'Sabana suroccidente': ['Bojacá  ', 'Funza ', 'Madrid ', 'Mosquera', 'Zipacón'],
                     'Soacha - Sibaté': ['Sibaté', 'Soacha'],
}

def load_semicolon_csv(path, columns=None, encodings=('utf-8', 'latin1')):
    last_error = None
    for encoding in encodings:
        try:
            if columns is None:
                return pd.read_csv(path, sep=';', encoding=encoding)
            return pd.read_csv(path, sep=';', encoding=encoding, usecols=list(columns))
        except Exception as exc:
            last_error = exc
    raise last_error

def load_transition(path):
    df = pd.read_csv(path, sep=',')
    df.columns = ['from_class', 'to_class', 'area_ha']
    df['area_ha'] = pd.to_numeric(df['area_ha'], errors='coerce')
    return df

def compute_transition_indices(path, municipio_base):
    df = load_transition(path)
    total = df['area_ha'].sum()
    f = df.set_index(['from_class', 'to_class'])['area_ha'].to_dict()
    get = lambda a, b: f.get((a, b), 0.0)
    F_ForAgro = get('Forest formation', 'Agricultural and livestock area')
    F_WatAgro = get('Water body', 'Agricultural and livestock area')
    F_AgroFor = get('Agricultural and livestock area', 'Forest formation')
    F_AgroWat = get('Agricultural and livestock area', 'Water body')
    F_ForFor = get('Forest formation', 'Forest formation')
    F_WatWat = get('Water body', 'Water body')
    F_AgroAgro = get('Agricultural and livestock area', 'Agricultural and livestock area')
    A_total = total
    degrad_total = (F_ForAgro + F_WatAgro) / A_total * 100 if A_total else np.nan
    recover_total = (F_AgroFor + F_AgroWat) / A_total * 100 if A_total else np.nan
    degrad_forest_rel = F_ForAgro / F_ForFor if F_ForFor else np.nan
    degrad_water_rel = F_WatAgro / F_WatWat if F_WatWat else np.nan
    recover_agro_rel = F_AgroFor / F_AgroAgro if F_AgroAgro else np.nan
    ind_neto_transicion = degrad_total - recover_total
    ind_neto_transicion_rel = (degrad_forest_rel + degrad_water_rel) - recover_agro_rel
    return {
        'municipio_base': municipio_base,
        'A_total': A_total,
        'F_ForAgro': F_ForAgro,
        'F_WatAgro': F_WatAgro,
        'F_AgroFor': F_AgroFor,
        'F_AgroWat': F_AgroWat,
        'F_ForFor': F_ForFor,
        'F_WatWat': F_WatWat,
        'F_AgroAgro': F_AgroAgro,
        'degrad_total': degrad_total,
        'recover_total': recover_total,
        'degrad_forest_rel': degrad_forest_rel,
        'degrad_water_rel': degrad_water_rel,
        'recover_agro_rel': recover_agro_rel,
        'ind_neto_transicion': ind_neto_transicion,
        'ind_neto_transicion_rel': ind_neto_transicion_rel,
    }


In [32]:
# Definir los períodos de transición y los años de interés
transition_periods = [
    (2011, 2014, 'Transition_*_2011-2014.csv'),
    (2014, 2017, 'Transition_*_2014-2017.csv'),
    (2018, 2021, 'Transition_*_2018-2021.csv'),
]

# Diccionarios para almacenar los resultados por año
data_2014 = {}
data_2017 = {}
data_2021 = {}

# Procesar cada período de transición
for start_year, end_year, pattern in transition_periods:
    print(f"Procesando período {start_year}-{end_year}...")
    transition_files = list(TRANSITIONS_DIR.glob(pattern))
    print(f"  Encontrados {len(transition_files)} archivos")
    
    for path in transition_files:
        # Extraer el nombre del municipio del archivo
        raw_name = path.name.replace('Transition_', '').replace(f'_{start_year}-{end_year}.csv', '')
        normalized = normalize_municipality(raw_name)
        municipio_base = manual_mapping.get(normalized, normalized)
        
        # Cargar la transición
        df_trans = load_transition(path)
        
        # Calcular área por clase para el año de inicio (from_class)
        area_from = df_trans.groupby('from_class')['area_ha'].sum()
        # Calcular área por clase para el año de fin (to_class)
        area_to = df_trans.groupby('to_class')['area_ha'].sum()
        # Área total (debe ser la misma para ambos años asumiendo conservación de área total)
        total_area = df_trans['area_ha'].sum()
        
        # Clases de interés
        classes_of_interest = [
            'Forest formation',
            'Agricultural and livestock area',
            'Non-vegetated area',
            'Water body'
        ]
        
        # Procesar según el año
        if start_year == 2011 and end_year == 2014:
            # Para 2011: usar área de inicio (from_class)
            if municipio_base not in data_2014:
                data_2014[municipio_base] = {'total_area': total_area}
            for cls in classes_of_interest:
                data_2014[municipio_base][cls] = area_from.get(cls, 0.0)
            # Para 2014: usar área de fin (to_class) del mismo período
            if municipio_base not in data_2017:
                data_2017[municipio_base] = {'total_area': total_area}
            for cls in classes_of_interest:
                data_2017[municipio_base][cls] = area_to.get(cls, 0.0)
        elif start_year == 2014 and end_year == 2017:
            # Para 2014: también podemos usar el año de inicio de 2014-2017 (debería ser consistente con el anterior)
            # Pero ya lo tenemos de 2011-2014, así que lo verificamos
            if municipio_base not in data_2017:
                data_2017[municipio_base] = {'total_area': total_area}
            for cls in classes_of_interest:
                # Si ya existe, promediamos? Pero asumimos consistencia.
                # Si no existe, lo establecemos.
                if cls not in data_2017[municipio_base]:
                    data_2017[municipio_base][cls] = area_from.get(cls, 0.0)
            # Para 2017: no lo necesitamos
        elif start_year == 2018 and end_year == 2021:
            # Para 2021: usar área de fin (to_class)
            if municipio_base not in data_2021:
                data_2021[municipio_base] = {'total_area': total_area}
            for cls in classes_of_interest:
                data_2021[municipio_base][cls] = area_to.get(cls, 0.0)
        else:
            print(f"    Advertencia: período {start_year}-{end_year} no procesado para años de interés")
    print(f"  Procesado {len(transition_files)} archivos para {start_year}-{end_year}")


Procesando período 2011-2014...
  Encontrados 22 archivos
  Procesado 22 archivos para 2011-2014
Procesando período 2014-2017...
  Encontrados 22 archivos
  Procesado 22 archivos para 2014-2017
Procesando período 2018-2021...
  Encontrados 22 archivos
  Procesado 22 archivos para 2018-2021


In [33]:
# Convertir los diccionarios a DataFrames y calcular porcentajes y WEI
def compute_percentages_and_wei(data_dict, year_label):
    """
    Convierte el diccionario de datos para un año en un DataFrame y calcula porcentajes y WEI.
    """
    records = []
    for municipio, values in data_dict.items():
        total_area = values['total_area']
        record = {'municipio_base': municipio, 'year': year_label, 'total_area': total_area}
        
        classes = ['Forest formation', 'Agricultural and livestock area', 'Non-vegetated area', 'Water body']
        areas = {}
        for cls in classes:
            area_val = values.get(cls, 0.0)
            areas[cls] = area_val
            record[f'{cls}_area'] = area_val
        
        # Calcular porcentajes
        for cls in classes:
            area_val = areas[cls]
            pct = (area_val / total_area * 100) if total_area > 0 else 0.0
            record[f'{cls}_perc'] = pct
        
        # Calcular WEI
        wei_weights = {
            'Forest formation': 1.0,
            'Agricultural and livestock area': 0.5,
            'Non-vegetated area': 0.0,
            'Water body': 1.0
        }
        wei = sum(areas[cls] * wei_weights[cls] for cls in classes) / total_area if total_area > 0 else 0.0
        record['WEI'] = wei
        
        records.append(record)
    
    df = pd.DataFrame(records)
    return df

# Procesar cada año
df_2014 = compute_percentages_and_wei(data_2014, 2014)
df_2017 = compute_percentages_and_wei(data_2017, 2017)
df_2021 = compute_percentages_and_wei(data_2021, 2021)

# Combinar todos los años
df_all = pd.concat([df_2014, df_2017, df_2021], ignore_index=True)

# Ordenar por municipio y año
df_all = df_all.sort_values(['municipio_base', 'year']).reset_index(drop=True)

# Mostrar una vista previa
print("DataFrame combinado (primeras 10 filas):")
display(df_all.head(10))

# Verificar que tenemos datos para los 21 municipios en cada año
print("\nConteo de municipios por año:")
print(df_all['year'].value_counts().sort_index())

DataFrame combinado (primeras 10 filas):


,municipio_base,year,total_area,Forest formation_area,Agricultural and livestock area_area,Non-vegetated area_area,Water body_area,Forest formation_perc,Agricultural and livestock area_perc,Non-vegetated area_perc,Water body_perc,WEI
0,Bojacá,2014,10231.053234,1694.900465,6521.247594,351.344646,12.296135,16.566236,63.739748,3.434100,0.120184,0.485563
1,Bojacá,2017,10231.053234,1634.485514,6451.213696,381.284885,11.939562,15.975731,63.055226,3.726741,0.116699,0.476200
2,Bojacá,2021,10231.053234,1616.397393,6527.582081,512.986977,8.999197,15.798934,63.801663,5.014019,0.087960,0.477877
3,Cajicá,2014,5131.741013,366.254465,3890.951583,791.340062,20.309459,7.137041,75.821277,15.420499,0.395762,0.454434
4,Cajicá,2017,5131.741013,338.196701,3846.589715,838.729741,34.472337,6.590292,74.956817,16.343961,0.671747,0.447404
5,Cajicá,2021,5131.741013,360.387121,3509.421439,1181.499268,19.240115,7.022707,68.386566,23.023361,0.374924,0.415909
6,Chía,2014,7992.731674,1471.743570,5028.658990,1226.772690,11.670121,18.413524,62.915399,15.348603,0.146009,0.500172
7,Chía,2017,7992.731674,1460.607514,4958.817849,1296.168966,25.656566,18.274197,62.041590,16.216846,0.320999,0.496160
8,Chía,2021,7992.731674,1454.639737,4609.964761,1612.952146,13.362679,18.199532,57.676961,20.180236,0.167185,0.472052
9,Cota,2014,5374.739420,965.039059,3555.629631,612.803628,23.165447,17.955086,66.154456,11.401550,0.431006,0.514633



Conteo de municipios por año:
year
2014    22
2017    22
2021    22
Name: count, dtype: int64


In [34]:
class_cols = {
    'Forest formation': 'Forest formation_perc',
    'Agricultural and livestock area': 'Agricultural and livestock area_perc',
    'Non-vegetated area': 'Non-vegetated area_perc',
    'Water body': 'Water body_perc',
    'WEI': 'WEI',
}

# Define line styles for each class
line_styles = {
    'Forest formation': 'solid',
    'Agricultural and livestock area': 'dash',
    'Non-vegetated area': 'dot',
    'Water body': 'dashdot',
    'WEI': 'solid',
}

df_plot = df_all[['municipio_base', 'year'] + list(class_cols.values())].rename(
    columns={v: k for k, v in class_cols.items()}
)

df_long = df_plot.melt(
    id_vars=['municipio_base', 'year'],
    var_name='class',
    value_name='value'
)

fig = go.Figure()

for (municipio, cls), group in df_long.groupby(['municipio_base', 'class']):
    fig.add_trace(
        go.Scatter(
            x=group['year'],
            y=group['value'],
            mode='lines+markers',
            name=f'{municipio} - {cls}',
            legendgroup=municipio,
            line=dict(
                dash=line_styles.get(cls, 'solid'),
                width=2,
            ),
            marker=dict(size=5),
            hovertemplate='<b>%{fullData.name}</b><br>Año: %{x}<br>Valor: %{y:.2f}<extra></extra>'
        )
    )

fig.update_layout(
    title='Evolución del uso del suelo y WEI por municipio',
    xaxis=dict(title='Año', dtick=1),
    yaxis=dict(title='Porcentaje / WEI'),
    legend=dict(traceorder='grouped', font=dict(size=9)),
    hovermode='closest',
    template='simple_white'
)

fig

## Uso suelo a través del tiempo (1985 - 2021)

### Revisión crecimiento urbano municipal

In [35]:
SOIL_PATH = DATA_DIR / 'Statistics-for-Website-MB-Cobertura-col3.xlsx'

In [36]:
# Funciones auxiliares (copiadas del notebook 30_transicion_pobreza_dispersion_adj.ipynb)
manual_mapping = {
    'bojaca': 'Bojacá',
    'cajica': 'Cajicá',
    'chia': 'Chía',
    'cota': 'Cota',
    'facatativa': 'Facatativá',
    'funza': 'Funza',
    'gachancipa': 'Gachancipá',
    'sibate': 'Sibaté',
    'sopo': 'Sopó',
    'tocancipa': 'Tocancipá',
    'zipacon': 'Zipacón',
    'zipaquira': 'Zipaquirá',
    'el rosal': 'El Rosal',
    'la calera': 'La Calera',
    'madrid': 'Madrid',
    'mosquera': 'Mosquera',
    'subachoque': 'Subachoque',
    'tabio': 'Tabio',
    'tenjo': 'Tenjo',
    'soacha': 'Soacha',
}

subregion_mapping = {'Guavio': ['La Calera'],
                     'Sabana Centro': ['Cajicá ', 'Chía ', 'Cota', 'Sopó ', 'Tabio ', 'Tenjo'],
                     'Sabana Norte': ['Gachancipá  ', 'Tocancipá  ', 'Zipaquirá'],
                     'Sabana noroccidente': ['El Rosal', 'Facatativá ', 'Subachoque'],
                     'Sabana suroccidente': ['Bojacá  ', 'Funza ', 'Madrid ', 'Mosquera', 'Zipacón'],
                     'Soacha - Sibaté': ['Sibaté', 'Soacha'],
}

In [37]:
soil = pd.read_excel(SOIL_PATH, sheet_name='COBERTURA_MUNICIPIO')
soil = soil[soil['departamento'].astype(str).str.strip().str.lower() == 'cundinamarca'].copy()

soil['municipio_norm'] = soil['municipio'].astype(str).apply(normalize_municipality)
mask = soil['municipio_norm'].isin(manual_mapping)
soil = soil[mask].copy()
soil['municipio'] = soil['municipio_norm'].map(manual_mapping)

rev_sub = {}
for subregion, muns in subregion_mapping.items():
    for m in muns:
        rev_sub[normalize_municipality(m.strip())] = subregion
soil['SUBREGIÓN'] = soil['municipio_norm'].map(rev_sub)

soil.head()

,municipio,departamento,pais,class_level_0,class_level_1,class_level_2,1985,1986,1987,1988,...,2017,2018,2019,2020,2021,2022,2023,2024,municipio_norm,SUBREGIÓN
4553,Bojacá,Cundinamarca,Colombia,Antrópico,3. Área agropecuaria,3.1. Silvicultura,583.480200,581.787511,569.134721,530.371816,...,773.367487,929.757179,891.352847,881.107092,854.286137,847.869546,794.939250,590.254727,bojaca,Sabana suroccidente
4554,Bojacá,Cundinamarca,Colombia,Antrópico,3. Área agropecuaria,3.4. Mosaico de agricultura o pasto,5124.834659,5072.531052,5368.899350,5278.280661,...,5555.141035,5549.612132,5588.552882,5612.968626,5688.797753,5598.267226,5454.451453,5275.260310,bojaca,Sabana suroccidente
4555,Bojacá,Cundinamarca,Colombia,Antrópico,4. Área sin vegetación,4.2. Infraestructura urbana,32.609970,33.233673,33.857377,35.104753,...,89.901577,95.514984,99.880866,107.632591,112.176719,129.907804,134.808287,136.412044,bojaca,Sabana suroccidente
4556,Bojacá,Cundinamarca,Colombia,Antrópico,4. Área sin vegetación,4.3. Minería,2.405972,2.405972,2.405972,3.207971,...,121.546814,126.804352,129.834116,138.477838,139.814521,144.359136,153.893910,154.606821,bojaca,Sabana suroccidente
4557,Bojacá,Cundinamarca,Colombia,Antrópico,4. Área sin vegetación,4.5. Otra área sin vegetación,503.277119,566.183655,475.743262,677.929924,...,263.846680,246.025407,256.807013,277.481000,267.500642,254.759117,331.034480,476.188992,bojaca,Sabana suroccidente


In [38]:
years = list(range(1985, 2022))
area_df = pd.DataFrame({'municipio': sorted(soil['municipio'].unique())})

for yr in years:
    sub = soil[['municipio', 'class_level_0', 'class_level_1', 'class_level_2', yr]].copy()
    sub.columns = ['municipio', 'class_level_0', 'class_level_1', 'class_level_2', 'area']
    sub['area'] = pd.to_numeric(sub['area'], errors='coerce').fillna(0)

    total_area = sub.groupby('municipio')['area'].sum().rename(f'area_total_ha_{yr}')

    lv0 = sub.groupby(['municipio', 'class_level_0'])['area'].sum().reset_index()
    lv0['share'] = lv0.groupby('municipio')['area'].transform(lambda s: s / s.sum())
    lv0_wide = lv0.pivot(index='municipio', columns='class_level_0', values='share')
    lv0_wide = lv0_wide.rename(columns={'Antrópico': f'share_anthropic_{yr}', 'Natural': f'share_natural_{yr}'})

    lv1 = sub.groupby(['municipio', 'class_level_1'])['area'].sum().reset_index()
    lv1['share'] = lv1.groupby('municipio')['area'].transform(lambda s: s / s.sum())
    lv1_wide = lv1.pivot(index='municipio', columns='class_level_1', values='share')
    lv1_wide = lv1_wide.rename(columns={
        '1. Formacion Boscosa': f'share_forest_{yr}',
        '2. Formación natural no boscosa': f'share_natural_non_forest_{yr}',
        '3. Área  agropecuaria': f'share_agriculture_{yr}',
        '4. Área sin vegetación': f'share_no_vegetation_{yr}',
        '5. Cuerpo de agua': f'share_water_{yr}',
    })

    sel2 = {
        f'share_urban_infrastructure_{yr}': '4.2. Infraestructura urbana',
        f'share_agriculture_pasture_mosaic_{yr}': '3.4. Mosaico de agricultura o pasto',
        f'share_forest_level2_{yr}': '1.1. Bosque',
        f'share_mining_{yr}': '4.3. Minería',
        f'share_other_no_vegetation_{yr}': '4.5. Otra área sin vegetación',
        f'share_herbazales_arbustales_{yr}': '2.7. Herbazales o arbustales andinos',
    }
    lv2 = sub.groupby(['municipio', 'class_level_2'])['area'].sum().reset_index()
    lv2['share'] = lv2.groupby('municipio')['area'].transform(lambda s: s / s.sum())
    lv2_wide = lv2.pivot(index='municipio', columns='class_level_2', values='share')
    lv2_feature_df = pd.DataFrame({feat: lv2_wide[cls].fillna(0) for feat, cls in sel2.items()})
    lv2_feature_df['municipio'] = lv2_wide.index

    lv0_total = sub.groupby(['municipio', 'class_level_0'])['area'].sum().reset_index()
    lv0_total_wide = lv0_total.pivot(index='municipio', columns='class_level_0', values='area')
    lv0_total_wide = lv0_total_wide.rename(columns={'Antrópico': f'total_anthropic_{yr}', 'Natural': f'total_natural_{yr}'})

    lv1_total = sub.groupby(['municipio', 'class_level_1'])['area'].sum().reset_index()
    lv1_total_wide = lv1_total.pivot(index='municipio', columns='class_level_1', values='area')
    lv1_total_wide = lv1_total_wide.rename(columns={
        '1. Formacion Boscosa': f'total_forest_{yr}',
        '2. Formación natural no boscosa': f'total_natural_non_forest_{yr}',
        '3. Área  agropecuaria': f'total_agriculture_{yr}',
        '4. Área sin vegetación': f'total_no_vegetation_{yr}',
        '5. Cuerpo de agua': f'total_water_{yr}',
    })

    sel2_total = {
        f'total_urban_infrastructure_{yr}': '4.2. Infraestructura urbana',
        f'total_agriculture_pasture_mosaic_{yr}': '3.4. Mosaico de agricultura o pasto',
        f'total_forest_level2_{yr}': '1.1. Bosque',
        f'total_mining_{yr}': '4.3. Minería',
        f'total_other_no_vegetation_{yr}': '4.5. Otra área sin vegetación',
        f'total_herbazales_arbustales_{yr}': '2.7. Herbazales o arbustales andinos',
        f'total_agua_{yr}': '5.1. Río, lago u océano',
    }
    lv2_total = sub.groupby(['municipio', 'class_level_2'])['area'].sum().reset_index()
    lv2_total_wide = lv2_total.pivot(index='municipio', columns='class_level_2', values='area')
    lv2_total_feature_df = pd.DataFrame({feat: lv2_total_wide[cls].fillna(0) for feat, cls in sel2_total.items()})
    lv2_total_feature_df['municipio'] = lv2_total_wide.index

    yr_df = total_area.reset_index().merge(lv0_wide.reset_index(), on='municipio', how='outer')
    yr_df = yr_df.merge(lv1_wide.reset_index(), on='municipio', how='outer')
    yr_df = yr_df.merge(lv2_feature_df.reset_index(drop=True), on='municipio', how='outer')
    yr_df = yr_df.merge(lv0_total_wide.reset_index(), on='municipio', how='outer')
    yr_df = yr_df.merge(lv1_total_wide.reset_index(), on='municipio', how='outer')
    yr_df = yr_df.merge(lv2_total_feature_df.reset_index(drop=True), on='municipio', how='outer')

    for col in yr_df.columns:
        if col != 'municipio' and col in area_df.columns:
            area_df.drop(columns=[col], inplace=True)
    area_df = area_df.merge(yr_df, on='municipio', how='left')

area_df.head()

,municipio,area_total_ha_1985,share_anthropic_1985,share_natural_1985,share_forest_1985,share_natural_non_forest_1985,share_agriculture_1985,share_no_vegetation_1985,share_water_1985,share_urban_infrastructure_1985,...,total_agriculture_2021,total_no_vegetation_2021,total_water_2021,total_urban_infrastructure_2021,total_agriculture_pasture_mosaic_2021,total_forest_level2_2021,total_mining_2021,total_other_no_vegetation_2021,total_herbazales_arbustales_2021,total_agua_2021
0,Bojacá,10243.435313,0.609816,0.390184,0.166602,0.222808,0.557266,0.052550,0.000774,0.003183,...,6543.083890,519.491882,8.999197,112.176719,5688.797753,1592.604647,139.814521,267.500642,0.000000,8.999197
1,Cajicá,5106.787314,0.829225,0.170775,0.134897,0.030105,0.763990,0.065235,0.005773,0.016396,...,3485.525902,1177.769017,18.972896,837.586622,3125.847686,363.771796,32.600952,307.581443,1.336067,18.972896
2,Chía,8007.127127,0.743043,0.256957,0.203466,0.046938,0.696716,0.046327,0.006553,0.018001,...,4621.220265,1617.495202,12.382700,1221.786880,4493.387902,1453.682113,43.025925,352.682397,49.888389,12.382700
3,Cota,5336.894291,0.739129,0.260871,0.211023,0.046459,0.723654,0.015475,0.003389,0.005693,...,3057.967143,1084.739198,10.424697,822.803510,2972.531145,887.708140,0.000000,261.935688,48.554224,10.424697
4,El Rosal,8692.095318,0.930240,0.069760,0.064769,0.000010,0.908338,0.021902,0.004981,0.006057,...,7394.698498,664.756484,58.706812,540.304221,7394.698498,573.933525,11.937422,112.514841,0.000000,58.706812


In [39]:
classes_level1 = ['total_forest', 'total_natural_non_forest', 'total_agriculture', 'total_no_vegetation', 'total_water']
wei_weights = {
    'total_forest': 0.3,
    'total_natural_non_forest': 0.25,
    'total_agriculture': 0.15,
    'total_no_vegetation': 0.0,
    'total_water': 0.3,
}

for yr in years:
    total_col = f'area_total_ha_{yr}'
    comp_cols = {cls: f'{cls}_{yr}' for cls in classes_level1}
    for col in comp_cols.values():
        if col not in area_df.columns:
            area_df[col] = 0.0
    weighted_sum = sum(area_df[comp_cols[cls]] * wei_weights[cls] for cls in classes_level1)
    area_df[f'WEI_{yr}'] = np.where(area_df[total_col] > 0, weighted_sum / area_df[total_col], 0.0)

In [40]:
for yr in range(1986, 2022):
    prev_yr = yr - 1

    start_col = f'total_urban_infrastructure_{prev_yr}'
    end_col = f'total_urban_infrastructure_{yr}'
    cagr_col = f'total_urban_infrastructure_CAGR_{yr}'
    area_df[cagr_col] = np.where(
        (area_df[start_col] > 0) & (area_df[end_col] > 0),
        (area_df[end_col] / area_df[start_col]) - 1,
        np.nan
    ) * 100

    start_col = f'total_agriculture_pasture_mosaic_{prev_yr}'
    end_col = f'total_agriculture_pasture_mosaic_{yr}'
    cagr_col = f'total_agriculture_pasture_mosaic_CAGR_{yr}'
    area_df[cagr_col] = np.where(
        (area_df[start_col] > 0) & (area_df[end_col] > 0),
        (area_df[end_col] / area_df[start_col]) - 1,
        np.nan
    ) * 100

    start_col = f'total_forest_level2_{prev_yr}'
    end_col = f'total_forest_level2_{yr}'
    cagr_col = f'total_forest_level2_CAGR_{yr}'
    area_df[cagr_col] = np.where(
        (area_df[start_col] > 0) & (area_df[end_col] > 0),
        (area_df[end_col] / area_df[start_col]) - 1,
        np.nan
    ) * 100

    start_col = f'total_agua_{prev_yr}'
    end_col = f'total_agua_{yr}'
    cagr_col = f'total_agua_CAGR_{yr}'
    area_df[cagr_col] = np.where(
        (area_df[start_col] > 0) & (area_df[end_col] > 0),
        (area_df[end_col] / area_df[start_col]) - 1,
        np.nan
    ) * 100

In [ ]:
# Calculation relative area and compound growth rate (CAGR) for urban infrastructure
def cal_cagr(x):
    return ((x['total_urban_infrastructure_2021'] / x['total_urban_infrastructure_1985'])**(1/(2021-1985)) - 1)*100

def cal_ar(x):
    return (x['total_urban_infrastructure_2021'] - x['total_urban_infrastructure_1985'])/(x['total_urban_infrastructure_1985']) * 100

print("Tasa de crecimiento anual compuesta terreno urbano (1985-2021)")
display(area_df.groupby('municipio').apply(cal_cagr).sort_values(ascending=False).reset_index().drop(columns=['level_1']).rename(columns={0: 'urban_cagr'}))
print("Porcentaje de crecimiento urbano (1985-2021)")
display(area_df.groupby('municipio').apply(cal_ar).sort_values(ascending=False).reset_index().drop(columns=['level_1']).rename(columns={0: 'urban_ar'}))

Tasa de crecimiento anual compuesta terreno urbano (1985-2021)


,municipio,urban_cagr
0,Gachancipá,10.465093
1,Cota,9.596564
2,Tocancipá,9.018404
3,Tenjo,8.098424
4,Sopó,7.700940
5,El Rosal,6.681691
6,Cajicá,6.606036
7,Chía,6.116714
8,Mosquera,5.803249
9,Funza,4.928701


Porcentaje de crecimiento urbano (1985-2021)


,municipio,urban_ar
0,Gachancipá,3498.340025
1,Cota,2608.256090
2,Tocancipá,2138.687741
3,Tenjo,1550.049779
4,Sopó,1345.112963
5,El Rosal,926.209660
6,Cajicá,900.333182
7,Chía,747.650119
8,Mosquera,662.016130
9,Funza,465.190333


### Crecimiento urbano anual

In [45]:
def build_long_for_index(value_col, label):
    records = []
    for _, row in area_df.iterrows():
        subregion = row['SUBREGIÓN']
        for yr in range(1985, 2022):
            val = row.get(f'{value_col}_{yr}')
            if not pd.isna(val):
                records.append({'municipio': row['municipio'], 'subregion': subregion, 'year': yr, 'value': val})
    return pd.DataFrame(records)

fig_wei = go.Figure()
wei_long = build_long_for_index('WEI', 'WEI')
for subregion, group in wei_long.groupby('subregion'):
    for municipio, mgroup in group.groupby('municipio'):
        fig_wei.add_trace(go.Scatter(x=mgroup['year'], y=mgroup['value'], mode='lines+markers', name=municipio, legendgroup=subregion, line=dict(width=2), marker=dict(size=4), visible='legendonly', hovertemplate='<b>%{fullData.name}</b><br>' + 'Año: %{x}<br>WEI: %{y:.3f}<extra></extra>'))

fig_wei.update_layout(title='Evolución WEI por municipio (1985-2021)', xaxis=dict(title='Año', dtick=1), yaxis=dict(title='WEI'), 
                      legend=dict(traceorder='grouped', font=dict(size=9)), 
                      template='simple_white',
                      width=1500,
                      height=700,
                      )

fig_wei

In [46]:
wei_long_all = build_long_for_index('WEI', 'WEI')
subregions = sorted(wei_long_all['subregion'].unique())
subregion_figs = {}

avg_wei = wei_long_all.groupby('year')['value'].mean().reset_index()
wei_global_min = wei_long_all['value'].min()
wei_global_max = wei_long_all['value'].max()
y_padding = (wei_global_max - wei_global_min) * 0.05 or 0.01
y_range = [wei_global_min - y_padding, wei_global_max + y_padding]

def resolve_label_overlaps(municipios, vals, min_gap):
    offsets = {m: 0.0 for m in municipios}
    for _ in range(10):
        shifted = {m: vals[m] + offsets[m] for m in municipios}
        sorted_m = sorted(municipios, key=lambda m: shifted[m])
        changed = False
        for i in range(len(sorted_m) - 1):
            m1, m2 = sorted_m[i], sorted_m[i + 1]
            if shifted[m2] - shifted[m1] < min_gap:
                if offsets[m1] >= 0:
                    offsets[m1] -= min_gap / 2
                else:
                    offsets[m1] -= min_gap / 2
                if offsets[m2] <= 0:
                    offsets[m2] += min_gap / 2
                else:
                    offsets[m2] += min_gap / 2
                changed = True
        if not changed:
            break
    return offsets

for subregion in subregions:
    sub_df = wei_long_all[wei_long_all['subregion'] == subregion]
    fig = go.Figure()

    municipios = []
    last_vals = {}
    for municipio, group in sub_df.groupby('municipio'):
        municipios.append(municipio)
        last_year = group['year'].max()
        last_val = group.loc[group['year'] == last_year, 'value'].iloc[0]
        last_vals[municipio] = last_val

    data_range = max(last_vals.values()) - min(last_vals.values()) if last_vals else 0.01
    min_gap = max(data_range * 0.02, 0.005)
    offsets = resolve_label_overlaps(municipios, last_vals, min_gap)

    for municipio, group in sub_df.groupby('municipio'):
        fig.add_trace(go.Scatter(
            x=group['year'],
            y=group['value'],
            mode='lines+markers',
            name=municipio,
            line=dict(width=2),
            marker=dict(size=4),
            hovertemplate='<b>%{fullData.name}</b><br>Año: %{x}<br>WEI: %{y:.3f}<extra></extra>',
            showlegend=False
        ))

        last_year = group['year'].max()
        last_val = group.loc[group['year'] == last_year, 'value'].iloc[0]
        fig.add_annotation(
            x=last_year + 0.3,
            y=last_val + offsets[municipio],
            text=municipio,
            showarrow=False,
            xanchor='left',
            yanchor='middle',
            font=dict(size=12)
        )

    fig.add_trace(go.Scatter(
        x=avg_wei['year'],
        y=avg_wei['value'],
        mode='lines',
        name='Promedio general',
        line=dict(color='black', width=3, dash='dash'),
        hovertemplate='<b>Mediana subregional</b><br>Año: %{x}<br>WEI: %{y:.3f}<extra></extra>'
    ))

    fig.update_layout(
        title=f'Evolución WEI por municipio - Subregión: {subregion} (1985-2021)',
        xaxis=dict(title='Año', dtick=5),
        yaxis=dict(title='WEI', range=y_range),
        template='simple_white',
        height=500,
        showlegend=True
    )

    subregion_figs[subregion] = fig

for subregion, fig in subregion_figs.items():
    # print(f"=== {subregion} ===")
    display(fig)


In [47]:
fig_urban_cagr = go.Figure()
urban_long = build_long_for_index('total_urban_infrastructure_CAGR', 'total_urban_infrastructure_CAGR')
urban_long = urban_long[urban_long['year']>2005]
for subregion, group in urban_long.groupby('subregion'):
    for municipio, mgroup in group.groupby('municipio'):
        fig_urban_cagr.add_trace(go.Scatter(x=mgroup['year'], y=mgroup['value'], mode='lines+markers', name=municipio, legendgroup=subregion, line=dict(width=2), marker=dict(size=4), visible='legendonly', hovertemplate='<b>%{fullData.name}</b><br>' + 'Año: %{x}<br>CAGR: %{y:.2f}%<extra></extra>'))

fig_urban_cagr.update_layout(title='CAGR Infraestructura urbana por municipio (2006-2021)', xaxis=dict(title='Año', dtick=1), yaxis=dict(title='CAGR (%)'), 
                             legend=dict(traceorder='grouped', font=dict(size=9)), template='simple_white',
                             width=1500,
                             height=700,
                             )
fig_urban_cagr

In [48]:
fig_agro_cagr = go.Figure()
agro_long = build_long_for_index('total_agriculture_pasture_mosaic_CAGR', 'total_agriculture_pasture_mosaic_CAGR')
for subregion, group in agro_long.groupby('subregion'):
    for municipio, mgroup in group.groupby('municipio'):
        fig_agro_cagr.add_trace(go.Scatter(x=mgroup['year'], y=mgroup['value'], mode='lines+markers', name=municipio, legendgroup=subregion, line=dict(width=2), marker=dict(size=4), visible='legendonly', hovertemplate='<b>%{fullData.name}</b><br>' + 'Año: %{x}<br>CAGR: %{y:.2f}%<extra></extra>'))

fig_agro_cagr.update_layout(title='CAGR Mosaico agricultura/pasto por municipio (1985-2021)', xaxis=dict(title='Año', dtick=1), yaxis=dict(title='CAGR (%)'), legend=dict(traceorder='grouped', font=dict(size=9)), template='simple_white')
fig_agro_cagr

In [49]:
fig_forest_cagr = go.Figure()
forest_long = build_long_for_index('total_forest_level2_CAGR', 'total_forest_level2_CAGR')
for subregion, group in forest_long.groupby('subregion'):
    for municipio, mgroup in group.groupby('municipio'):
        fig_forest_cagr.add_trace(go.Scatter(x=mgroup['year'], y=mgroup['value'], mode='lines+markers', name=municipio, legendgroup=subregion, line=dict(width=2), marker=dict(size=4), visible='legendonly', hovertemplate='<b>%{fullData.name}</b><br>' + 'Año: %{x}<br>CAGR: %{y:.2f}%<extra></extra>'))

fig_forest_cagr.update_layout(title='CAGR Bosque por municipio (1985-2021)', xaxis=dict(title='Año', dtick=1), yaxis=dict(title='CAGR (%)'), legend=dict(traceorder='grouped', font=dict(size=9)), template='simple_white')
fig_forest_cagr

In [50]:
fig_agua_cagr = go.Figure()
agua_long = build_long_for_index('total_agua_CAGR', 'total_agua_CAGR')
for subregion, group in agua_long.groupby('subregion'):
    for municipio, mgroup in group.groupby('municipio'):
        fig_agua_cagr.add_trace(go.Scatter(x=mgroup['year'], y=mgroup['value'], mode='lines+markers', name=municipio, legendgroup=subregion, line=dict(width=2), marker=dict(size=4), visible='legendonly', hovertemplate='<b>%{fullData.name}</b><br>' + 'Año: %{x}<br>CAGR: %{y:.2f}%<extra></extra>'))

fig_agua_cagr.update_layout(title='CAGR Agua por municipio (1985-2021)', xaxis=dict(title='Año', dtick=1), yaxis=dict(title='CAGR (%)'), legend=dict(traceorder='grouped', font=dict(size=9)), template='simple_white')
fig_agua_cagr


,municipio,total_urban_infrastructure_CAGR_1986,total_urban_infrastructure_CAGR_1987,total_urban_infrastructure_CAGR_1988,total_urban_infrastructure_CAGR_1989,total_urban_infrastructure_CAGR_1990,total_urban_infrastructure_CAGR_1991,total_urban_infrastructure_CAGR_1992,total_urban_infrastructure_CAGR_1993,total_urban_infrastructure_CAGR_1994,...,total_urban_infrastructure_CAGR_2012,total_urban_infrastructure_CAGR_2013,total_urban_infrastructure_CAGR_2014,total_urban_infrastructure_CAGR_2015,total_urban_infrastructure_CAGR_2016,total_urban_infrastructure_CAGR_2017,total_urban_infrastructure_CAGR_2018,total_urban_infrastructure_CAGR_2019,total_urban_infrastructure_CAGR_2020,total_urban_infrastructure_CAGR_2021
0,Bojacá,1.912615,1.876723,3.684205,2.284291,2.233315,1.941803,0.238094,2.137816,1.162781,...,2.914855,3.594770,1.997918,0.309294,2.363874,1.305223,6.243948,4.570887,7.760970,4.221888
1,Cajicá,11.489131,10.305174,5.882383,2.859519,3.018208,0.308410,1.229848,1.442638,0.673633,...,5.586875,2.161440,2.129399,3.403315,3.655553,5.346435,2.287341,4.647065,1.402338,3.204930
2,Chía,5.562394,7.259992,2.947571,4.294797,1.830200,1.647535,4.322220,0.188323,0.046993,...,3.461025,2.593895,1.394943,1.590731,1.498101,2.376586,2.142226,4.521532,1.907375,2.680214
3,Cota,1.466153,53.468356,10.734551,24.149990,15.616489,7.109038,14.159456,12.209620,16.580659,...,10.355692,6.700545,5.416614,4.248916,3.615430,2.339260,3.805350,6.445808,2.276581,4.350172
4,El Rosal,3.214883,57.540444,14.151843,37.921269,16.324890,4.772650,12.093140,2.660821,13.760489,...,0.975059,1.095637,1.138868,1.107884,1.041852,0.728886,2.488544,1.343222,0.934577,2.104377
5,Facatativá,6.022607,10.295867,5.293204,6.250021,5.466699,1.364033,3.857644,8.062158,4.129994,...,1.495806,1.462420,2.491626,1.580731,3.477122,1.503834,2.666790,3.353887,2.503645,2.367308
6,Funza,2.766091,7.215014,6.380769,4.391991,5.776919,2.849575,5.454528,4.679845,5.254728,...,7.112405,5.715344,4.207876,5.313933,3.132731,1.398979,1.897994,3.164228,2.922113,3.649354
7,Gachancipá,19.354916,35.135339,15.000027,0.000000,0.000000,5.217429,0.826459,0.000000,0.000000,...,4.674319,6.149286,4.206891,12.111021,5.017724,5.059001,6.527499,5.675494,4.420177,1.547577
8,La Calera,0.000000,4.566256,3.930153,2.100859,0.823042,3.673442,0.000000,31.102197,0.000000,...,2.791234,1.416774,4.423791,2.898587,3.900343,3.858140,3.212805,3.501925,2.443583,7.431289
9,Madrid,7.404058,9.419636,10.369782,15.759345,11.714876,3.372139,6.020336,3.689785,4.812923,...,3.179683,3.356752,1.614798,1.595078,2.319887,1.459988,1.721134,3.988626,2.043193,5.656539


In [51]:
def plot_subregion_variable(value_col, y_label=None):
    long_df = build_long_for_index(value_col, value_col)
    long_df = long_df[long_df['year']>2005]
    subregions = sorted(long_df['subregion'].unique())
    figs = {}

    med = long_df.groupby('year')['value'].median().reset_index()
    global_min = long_df['value'].min()
    global_max = long_df['value'].max()
    y_padding = (global_max - global_min) * 0.05 or 0.01
    y_range = [global_min - y_padding, global_max + y_padding]

    def resolve_label_overlaps(municipios, vals, min_gap):
        offsets = {m: 0.0 for m in municipios}
        for _ in range(10):
            shifted = {m: vals[m] + offsets[m] for m in municipios}
            sorted_m = sorted(municipios, key=lambda m: shifted[m])
            changed = False
            for i in range(len(sorted_m) - 1):
                m1, m2 = sorted_m[i], sorted_m[i + 1]
                if shifted[m2] - shifted[m1] < min_gap:
                    offsets[m1] -= min_gap / 2
                    offsets[m2] += min_gap / 2
                    changed = True
            if not changed:
                break
        return offsets

    for subregion in subregions:
        sub_df = long_df[long_df['subregion'] == subregion]
        fig = go.Figure()

        municipios = []
        last_vals = {}
        for municipio, group in sub_df.groupby('municipio'):
            municipios.append(municipio)
            last_year = group['year'].max()
            last_val = group.loc[group['year'] == last_year, 'value'].iloc[0]
            last_vals[municipio] = last_val

        data_range = max(last_vals.values()) - min(last_vals.values()) if last_vals else 0.01
        min_gap = max(data_range * 0.05, 0.005)
        offsets = resolve_label_overlaps(municipios, last_vals, min_gap)

        for municipio, group in sub_df.groupby('municipio'):
            fig.add_trace(go.Scatter(
                x=group['year'],
                y=group['value'],
                mode='lines+markers',
                name=municipio,
                line=dict(width=2),
                marker=dict(size=4),
                hovertemplate=f'<b>%{{fullData.name}}</b><br>Año: %{{x}}<br>{y_label or value_col}: %{{y:.3f}}<extra></extra>',
                showlegend=False
            ))

            last_year = group['year'].max()
            last_val = group.loc[group['year'] == last_year, 'value'].iloc[0]
            if municipio in ('Cota', 'Sopó', 'Facatativá', 'Sibaté', 'Tenjo'):
                last_val += 0.75
            if municipio in ('Chía', 'Gachancipá', 'Mosquera', 'Soacha'):
                last_val -= 0.75
            if municipio in ('Bojacá'):
                last_val += 0.25
            if municipio in ('Funza'):
                last_val -= 0.25
            fig.add_annotation(
                x=last_year + 0.3,
                y=last_val, # + offsets[municipio],
                text=municipio,
                showarrow=False,
                xanchor='left',
                yanchor='middle',
                font=dict(size=12)
            )

        fig.add_trace(go.Scatter(
            x=med['year'],
            y=med['value'],
            mode='lines',
            name='Mediana subregional',
            line=dict(color='black', width=3, dash='dash'),
            hovertemplate=f'<b>Mediana subregional</b><br>Año: %{{x}}<br>{y_label or value_col}: %{{y:.3f}}<extra></extra>'
        ))

        fig.update_layout(
            title=f'Evolución {y_label or value_col} por municipio - Subregión: {subregion} (1985-2021)',
            xaxis=dict(title='Año', dtick=5),
            yaxis=dict(title=y_label or value_col, range=y_range),
            template='simple_white',
            height=500,
            showlegend=True
        )

        figs[subregion] = fig

    for subregion, fig in figs.items():
        display(fig)

    return figs

plot_subregion_variable('total_urban_infrastructure_CAGR', y_label=None)

{'Guavio': Figure({
     'data': [{'hovertemplate': ('<b>%{fullData.name}</b><br>Año' ... '_CAGR: %{y:.3f}<extra></extra>'),
               'line': {'width': 2},
               'marker': {'size': 4},
               'mode': 'lines+markers',
               'name': 'La Calera',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': '1gfXB9gH2QfaB9sH3AfdB94H3wfgB+EH4gfjB+QH5Qc=', 'dtype': 'i2'},
               'y': {'bdata': ('MMJW1Kmd+D/y/DPEQlgJQDdxdfgpBR' ... 'A88QMMQLaV+kR1jANAjVoX4KO5HUA='),
                     'dtype': 'f8'}},
              {'hovertemplate': ('<b>Mediana subregional</b><br>' ... '_CAGR: %{y:.3f}<extra></extra>'),
               'line': {'color': 'black', 'dash': 'dash', 'width': 3},
               'mode': 'lines',
               'name': 'Mediana subregional',
               'type': 'scatter',
               'x': {'bdata': '1gfXB9gH2QfaB9sH3AfdB94H3wfgB+EH4gfjB+QH5Qc=', 'dtype': 'i2'},
               'y': {'bdata': ('tUDC9